# Modeling and Evaluation

In this section, we will build and evaluate our machine learning models. We will use the processed data from the previous steps and apply various modeling techniques to find the best-performing model for our task.


# Table of Contents

Take this as an example for a Table of Contents for your notebook.
We have to fix all the names and sections according to what we actually do in the notebook.

<a class="anchor" id="top"></a>

** **

1. [Importing Libraries & Data](#1.-Importing-Libraries-&-Data) <br><br>
    
2. [Exploratory Data Analysis](#2.-Exploratory-Data-Analysis)
    
   2.1 [Incoherencies](#2.1-Incoherencies) <br>
   
   &emsp; 2.1.1 [Address Identified Incoherencies](#2.1.1-Address-Identified-Incoherencies) <br><br>
    
3. [Data Cleaning & Preprocessing](#3.-Data-Cleaning-&-Preprocessing)

   3.1 [Duplicates](#3.1-Duplicates) <br>
    
   3.2 [Feature Engineering](#3.2-Feature-Engineering) <br>
   
   &emsp; 3.2.1 [Data Type Conversions](#3.2.1-Data-Type-Conversions) <br>
   
   &emsp; 3.2.2 [Encoding](#3.2.2-Encoding) <br>
   
   &emsp; 3.2.3 [Other Transformations](#3.2.3-Other-Transformations) <br>
    
   &emsp; 3.2.4 [Unique Feature-Pair Analysis](#3.2.4-Unique-Feature-Pair-Analysis) <br> 

   3.3 [Train-Test Split](#3.3-Train-Test-Split) <br>
   
   3.4 [Missing Values](#3.4-Missing-Values) <br>
    
   3.5 [Outliers](#3.5-Outliers) <br>

   3.6 [Visualisations](#3.6-Visualisations) <br><br>
   

<a id="sec-1-imports"></a>
## 1. Import Libraries


In [1]:
# --- Imports ---
import os
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.base import clone


import os
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.metrics import mean_squared_error, r2_score




<a id="sec-2-load"></a>
## 2. Load Dataset


In [2]:

# --- Define paths ---
data_dir = "../data/"
encoded_dir = os.path.join(data_dir, "feature_selection")

# --- Load encoded & scaled datasets ---
X_train = pd.read_csv(os.path.join(encoded_dir, "20_X_train.csv"))
y_train = pd.read_csv(os.path.join(encoded_dir, "20_y_train.csv")).squeeze()

X_val = pd.read_csv(os.path.join(encoded_dir, "20_X_val.csv"))
y_val = pd.read_csv(os.path.join(encoded_dir, "20_y_val.csv")).squeeze()

X_test = pd.read_csv(os.path.join(encoded_dir, "20_X_test.csv"))

# --- Sanity checks ---
print("Encoded datasets successfully loaded!")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}")



Encoded datasets successfully loaded!
X_train shape: (56979, 12)
y_train shape: (56979,)
X_val shape:   (18994, 12)
y_val shape:   (18994,)
X_test shape:  (32567, 12)


In [3]:
# ======================================================
# Utilities: metrics & evaluation
# ======================================================

def evaluate_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))  # compatible with older sklearn versions
    r2 = float(r2_score(y_true, y_pred))
    return {"RMSE": rmse, "R2": r2}

def report_model(name, y_true, y_pred):
    m = evaluate_regression(y_true, y_pred)
    print(f"[{name}]  RMSE: {m['RMSE']:,.2f} | R²: {m['R2']:.4f}")
    return m


In [4]:
# ======================================================
# Baseline Model 1 — Linear Regression
# ======================================================
from sklearn.linear_model import LinearRegression

lin = LinearRegression()
lin.fit(X_train, y_train)

y_val_pred_lin = lin.predict(X_val)
metrics_lin = report_model("LinearRegression (baseline)", y_val, y_val_pred_lin)


[LinearRegression (baseline)]  RMSE: 6,073.99 | R²: 0.5997


In [5]:
# ======================================================
# Baseline Model 2 — RandomForest
# ======================================================
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_val_pred_rf = rf.predict(X_val)
metrics_rf = report_model("RandomForest (baseline)", y_val, y_val_pred_rf)


[RandomForest (baseline)]  RMSE: 3,799.57 | R²: 0.8434


In [6]:
# ======================================================
# Compare baselines & pick current best
# ======================================================
import pandas as pd

cmp = pd.DataFrame([
    {"model": "LinearRegression", **metrics_lin},
    {"model": "RandomForest",     **metrics_rf},
]).sort_values(by="RMSE")

display(cmp)

best_name = cmp.iloc[0]["model"]
print(f"🏆 Current best (validation): {best_name}")


,model,RMSE,R2
1,RandomForest,3799.572676,0.843366
0,LinearRegression,6073.993183,0.599719


🏆 Current best (validation): RandomForest


In [7]:
# ======================================================
# Compare baselines & pick current best
# ======================================================
import pandas as pd

cmp = pd.DataFrame([
    {"model": "LinearRegression", **metrics_lin},
    {"model": "RandomForest",     **metrics_rf},
]).sort_values(by="RMSE")

display(cmp)

best_name = cmp.iloc[0]["model"]
print(f"🏆 Current best (validation): {best_name}")


,model,RMSE,R2
1,RandomForest,3799.572676,0.843366
0,LinearRegression,6073.993183,0.599719


🏆 Current best (validation): RandomForest


In [ ]:
# ======================================================
# 🎯 Create Kaggle Submission (carID, price)
# ======================================================

# 1) Load the raw test file to obtain carID
test_raw_path = os.path.join(data_dir, "test.csv")
test_raw = pd.read_csv(test_raw_path)

assert "carID" in test_raw.columns, "carID not found in ../data/test.csv"
assert len(test_raw) == len(X_test), "Length mismatch between test.csv and X_test_final!"

car_ids = test_raw["carID"].reset_index(drop=True)

# 2) Ensure a best_model exists (fallback if cell 7 was not executed)
try:
    best_model
except NameError:
    if "rf" in globals():
        best_model = rf
        print("ℹ️ Using RandomForest as best_model (fallback).")
    elif "lin" in globals():
        best_model = lin
        print("ℹ️ Using LinearRegression as best_model (fallback).")
    else:
        raise RuntimeError("No trained model found. Please run the training cells first.")

# 3) Generate predictions
y_test_pred = best_model.predict(X_test)

# (Optional) Round/clip according to competition rules
# Here we round to whole units, as in the sample, without allowing negative prices:
y_test_pred = np.clip(y_test_pred, a_min=0, a_max=None)
y_test_pred_rounded = np.rint(y_test_pred).astype(int)

# 4) Build the submission DataFrame
submission = pd.DataFrame({
    "carID": car_ids,
    "price": y_test_pred_rounded  # optionally switch to y_test_pred if floats are allowed or preferred
})

# 5) Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))


ℹ️ Using RandomForest as best_model (fallback).
✅ Submission saved to: ../data/submissions/30_submission_20251031_1447.csv


,carID,price
0,89856,19530
1,106581,19454
2,80886,19454
3,100174,19454
4,81376,19530
5,85391,19454
6,82175,19454
7,95250,19530
8,85071,19530
9,96210,19530


Pipeline to the Kaggle Competition Submission

- you need to set up your api key in the .env file as KAGGLE_USERNAME and KAGGLE_KEY

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os, json

env_path = Path("..") / ".env"   
load_dotenv(env_path, override=True)

kuser = os.getenv("KAGGLE_USERNAME")
kkey  = os.getenv("KAGGLE_KEY")

print("KAGGLE_USERNAME:", kuser)
print(".env loaded from:", env_path.resolve())


KAGGLE_USERNAME: mehmet1700
.env loaded from: /Users/karaca/src/MachineLearningProject-NOVAIMS2025/.env


In [51]:
!kaggle competitions submit -c cars4you -f {sub_path} -m "Message"


100%|█████████████████████████████████████████| 390k/390k [00:00<00:00, 416kB/s]
Successfully submitted to Cars4you